# 01 · comma2k19 preprocessing smoke test

Processes only **two segments** from the first archive.
Inspect synchronization before processing the full dataset.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

# ============================================================
# Repository
# ============================================================
REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "--branch", BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "origin", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "checkout", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

# ============================================================
# Paths
# ============================================================
DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"

COMMA_ROOT = DATA_ROOT / "comma2k19"
RAW_ROOT = COMMA_ROOT / "raw"
PROCESSED_ROOT = COMMA_ROOT / "processed" / "v1"

MANIFEST_ROOT = DRIVE_ROOT / "manifests" / "stage3" / "v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"

for p in [PROCESSED_ROOT, MANIFEST_ROOT, OUTPUT_ROOT, PRETRAINED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# ============================================================
# Install project from the existing pyproject.toml
# --no-deps keeps Colab's/DACON's binary stack intact.
# ============================================================
subprocess.run(
    [
        sys.executable,
        "-m", "pip", "install",
        "-q", "--no-deps", "-e", str(REPO),
    ],
    check=True,
)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print("Repository     :", REPO)
print("Branch         :", BRANCH)
print("Commit         :", commit)
print("RAW_ROOT       :", RAW_ROOT)
print("PROCESSED_ROOT :", PROCESSED_ROOT)

In [ ]:
import pandas as pd
from blackbox_detection.stage3.comma2k19 import (
    find_archives,
    PrepareConfig,
    prepare_archive,
)

archives = find_archives(RAW_ROOT)
assert archives, f"No archives found under {RAW_ROOT}"

cfg = PrepareConfig(processed_root=PROCESSED_ROOT, overwrite=False)
report = prepare_archive(archives[0], cfg, max_segments=2)
display(report)

assert "error" not in report.columns or report["error"].isna().all(), report

In [ ]:
import matplotlib.pyplot as plt
from blackbox_detection.stage3.schema import read_frame_table

row = report.iloc[0]
meta = read_frame_table(PROCESSED_ROOT / row.metadata_relpath)

print("metadata shape:", meta.shape)
display(meta.head())

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
t = meta.timestamp - meta.timestamp.iloc[0]

axes[0].plot(t, meta.speed_mps)
axes[0].set_ylabel("speed m/s")

axes[1].plot(t, meta.accel_from_speed_mps2)
axes[1].set_ylabel("accel m/s²")

axes[2].plot(t, meta.steering_deg)
axes[2].set_ylabel("steer deg")

axes[3].plot(t, meta.yaw_rate_rps)
axes[3].set_ylabel("yaw rad/s")
axes[3].set_xlabel("seconds")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt

video_path = PROCESSED_ROOT / row.video_relpath
cap = cv2.VideoCapture(str(video_path))

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
duration = frame_count / fps if fps > 0 else float("nan")

print("video path :", video_path)
print("frames     :", frame_count)
print("fps        :", fps)
print("duration s :", duration)
print("metadata   :", len(meta))

# Video and metadata should correspond frame-by-frame after 20 -> 10 Hz conversion.
assert abs(fps - 10.0) < 0.05, fps
assert abs(frame_count - len(meta)) <= 1, (frame_count, len(meta))

indices = sorted(set(
    i for i in [0, 150, 300, 450, 599]
    if 0 <= i < frame_count and i < len(meta)
))

for idx in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, bgr = cap.read()
    if not ok:
        continue

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 5))
    plt.imshow(rgb)
    plt.title(
        f"frame {idx} | "
        f"speed={meta.speed_mps.iloc[idx]:.1f} m/s | "
        f"steer={meta.steering_deg.iloc[idx]:.1f}°"
    )
    plt.axis("off")
    plt.show()

cap.release()

**Stop here and inspect.** The video should be 10 Hz and metadata should have
essentially the same number of rows. Steering/yaw changes should be temporally
plausible. Only after this check should notebook 02 be run.